# 03. Data Preprocessing Pipeline
**Project:** Academic Sports Machine Learning Project  
**Domain:** Cricket Player Performance Analytics  
**Course Context:** Module 2 (Scaling, Standardization, and Encoding)  

---

## 1. Objectives & Academic Methodology
1. Implement **Train/Test separation BEFORE any transformation** to guarantee zero data leakage.
2. Eliminate non-predictive identifiers (`Player_ID`, `Player_Name`, `Match_ID`).
3. Extract date features from `Match_Date`.
4. Treat `Strike_Rate` outliers with domain-justified clipping.
5. Encode nominal categorical features using `OneHotEncoder`.
6. Standardize numerical features using `StandardScaler` (fit on train, transform test).
7. Persist preprocessed datasets and pipeline artifacts for downstream reuse.


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from src.preprocessing import SportsDataPreprocessor, create_binary_target, split_data, load_raw_data
from src.utils import DATA_PROCESSED_DIR, MODELS_DIR, save_artifact, set_seed

set_seed(42)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = load_raw_data()
print(f"Raw dataset loaded: {df.shape}")

Raw dataset loaded: (10000, 31)


## 2. Target Creation & Separation
We isolate the target `High_Performer` and strictly remove `Performance_Score` from features.

In [ ]:
X, y = create_binary_target(df, threshold=60.0)
print(f"Features shape: {X.shape}, Target shape: {y.shape}")
print(f"Positive class ratio: {y.mean():.4f}")
assert 'Performance_Score' not in X.columns, "Leakage check failed: Performance_Score still in X!"
print("[VERIFIED] Zero target leakage: Performance_Score removed.")

Features shape: (10000, 30), Target shape: (10000,)
Positive class ratio: 0.1904
[VERIFIED] Zero target leakage: Performance_Score removed.


## 3. Stratified Train/Test Split (80/20)
Split is executed prior to any scaling or encoding.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
print(f"Train set: {X_train.shape[0]} samples (Class 1: {y_train.mean():.4f})")
print(f"Test set:  {X_test.shape[0]} samples (Class 1: {y_test.mean():.4f})")
assert abs(y_train.mean() - y_test.mean()) < 0.005, "Stratification mismatch!"
print("[VERIFIED] Stratification successfully preserved target proportions.")

Train set: 8000 samples (Class 1: 0.1904)
Test set:  2000 samples (Class 1: 0.1905)
[VERIFIED] Stratification successfully preserved target proportions.


## 4. Fitting Preprocessor on Training Data
The custom `SportsDataPreprocessor` drops identifiers, extracts dates, clips strike rates, one-hot encodes categoricals, and scales numerical columns.

In [ ]:
preprocessor = SportsDataPreprocessor(sr_cap=400.0)
preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Processed Train Shape: {X_train_proc.shape}")
print(f"Processed Test Shape:  {X_test_proc.shape}")
print(f"Total features generated: {len(preprocessor.get_feature_names_out())}")

Processed Train Shape: (8000, 82)
Processed Test Shape:  (2000, 82)
Total features generated: 82


## 5. Inspecting Preprocessed Features
Verifying feature names and scaling statistics.

In [ ]:
print("Sample preprocessed features:")
print(X_train_proc.iloc[:3, :8])

# Verify mean ~ 0 and std ~ 1 on training set numeric features
sample_num_cols = ['Runs', 'Batting_Average', 'Wickets', 'Economy']
print("Train numeric feature scale verification (mean, std):")
for col in sample_num_cols:
    print(f"{col:20s} Mean={X_train_proc[col].mean():.3f}, Std={X_train_proc[col].std():.3f}")

Sample preprocessed features:
           Age  Experience_Years  Matches_Played      Runs  Balls_Faced  \
8965  0.888177          1.435343        1.402670 -0.763396    -1.583298   
9784  1.340367          1.584401        0.516278  1.449489     1.246509   
1145  1.491097          0.839108        0.203433  0.422078     0.528498   

         Fours     Sixes  Strike_Rate  
8965 -1.279995 -1.048861     1.728335  
9784  1.999565  0.823437    -0.094720  
1145  0.359785  0.448977    -0.254836  
Train numeric feature scale verification (mean, std):
Runs                 Mean=-0.000, Std=1.000
Batting_Average      Mean=-0.000, Std=1.000
Wickets              Mean=-0.000, Std=1.000


Economy              Mean=0.000, Std=1.000


## 6. Persisting Preprocessed Datasets & Pipeline Artifacts

In [ ]:
# Save datasets
X_train_proc.to_csv(DATA_PROCESSED_DIR / "train_features.csv", index=False)
X_test_proc.to_csv(DATA_PROCESSED_DIR / "test_features.csv", index=False)
y_train.to_frame(name='High_Performer').to_csv(DATA_PROCESSED_DIR / "train_target.csv", index=False)
y_test.to_frame(name='High_Performer').to_csv(DATA_PROCESSED_DIR / "test_target.csv", index=False)

# Save pipeline artifact
save_artifact(preprocessor, MODELS_DIR / "preprocessor.joblib")
print("[SUCCESS] All preprocessed datasets and pipeline artifacts saved!")

[SUCCESS] Artifact saved to: C:\ML_project\models\preprocessor.joblib
[SUCCESS] All preprocessed datasets and pipeline artifacts saved!


## 7. Academic Viva Questions & Defense Notes

### Q1: Why must the scaler be fitted ONLY on the training split?
**Defense:** Fitting the scaler on the entire dataset before splitting causes information leakage (distribution parameters $\mu$ and $\sigma$ from test data leak into training). In production, future test instances are unseen; fitting on test data violates the fundamental assumption of empirical risk minimization.

### Q2: Why use One-Hot Encoding instead of Label Encoding for Team and Venue?
**Defense:** `Team`, `Opponent`, and `Venue` are nominal categorical variables with no inherent natural ordering. Label encoding would introduce an arbitrary ordinal relationship ($0 < 1 < 2$), misleading linear models and distance-based metrics into assuming mathematical distances between categorical teams.
